# Lecture 2.2 — ModelSettings: temperature, top_p, tool_choice, max_tokens

**Course:** OpenAI Agents SDK — Complete Course  
**Section:** 02 — Agents: Configuration & Behaviour

In this notebook we do a full deep-dive on `ModelSettings` — the dataclass that controls how the underlying language model behaves when an agent makes a call. We cover every field, organise them into tiers by how commonly you'll use them, and demonstrate the most important ones in live code.

By the end of this notebook you will:
- Understand every `ModelSettings` field and when to use it
- Know how `temperature`, `max_tokens`, `reasoning`, and `verbosity` interact on GPT-5 models
- Understand how agent-level and `RunConfig`-level settings layer using `ModelSettings.resolve()`

## Cell 1 — Install the SDK

We install the `openai-agents` package before anything else. The `-q` flag suppresses verbose pip output to keep the notebook readable.

> **Already installed?** If you are running this in a Colab session where you installed the package earlier, pip will confirm it is present and move on.

In [1]:
!pip install openai-agents -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.8/850.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 7.1 MB/s eta 0:00:00


## Cell 2 — API Key Setup (Google Colab Secrets)

We retrieve the OpenAI API key from Colab's **Secrets** vault and write it to the environment so the SDK picks it up automatically.

**How to add your key in Colab:**
1. Click the **🔑 key icon** in the left sidebar ("Secrets").
2. Click **+ Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Enable the toggle so this notebook can access it.
5. Run this cell.

> **Running locally?** Skip the `userdata` import and instead set the variable in your terminal before launching Jupyter:  
> `export OPENAI_API_KEY="sk-..."` (macOS/Linux) or `set OPENAI_API_KEY=sk-...` (Windows CMD).

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Imports

We import three things for this lecture:

| Import | Source | Purpose |
|---|---|---|
| `Reasoning` | `openai.types.shared` | Controls GPT-5 reasoning effort |
| `Agent` | `agents` | The agent class |
| `ModelSettings` | `agents` | The settings dataclass |
| `Runner` | `agents` | Executes agent runs |

**Important:** `Reasoning` comes from `openai.types.shared` — not from the `agents` package. It is an OpenAI SDK type that the Agents SDK adopts directly. This import will appear in every GPT-5 configuration going forward, so it is worth committing to memory.

`RunConfig` is also imported here because we use it in the `resolve()` demonstration cell later.

In [3]:
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, RunConfig, Runner

## Cell 4 — Full ModelSettings Reference (Conceptual)

Before writing a single line of agent code, it is worth understanding the full surface area of `ModelSettings`. The fields are organised into three tiers based on how frequently you will encounter them.

---

### Tier 1 — Everyday Tuning (works across most models)

| Field | Type | What it controls |
|---|---|---|
| `temperature` | `float \| None` | Randomness of output. `0.0` = near-deterministic; higher values add creativity. `None` defers to model default. |
| `top_p` | `float \| None` | Nucleus sampling threshold. Limits token selection to the top cumulative probability mass. Often left at `None`. |
| `frequency_penalty` | `float \| None` | Penalises tokens proportional to how often they have appeared. Reduces repetition in long outputs. Range: `0.0–2.0`. |
| `presence_penalty` | `float \| None` | Penalises tokens that have appeared at all. Encourages the model to introduce new topics. Range: `0.0–2.0`. |
| `max_tokens` | `int \| None` | Maximum number of output tokens to generate. The model stops generating after this limit. |
| `tool_choice` | `"auto" \| "required" \| "none" \| str \| None` | Controls whether/which tools the model must call. `"auto"` = model decides; `"required"` = must call a tool; `"none"` = must not call any tool. |
| `parallel_tool_calls` | `bool \| None` | Allow multiple tool calls in one turn. `None` defers to provider default (typically `True`). |
| `truncation` | `"auto" \| "disabled" \| None` | Context overflow strategy. `"auto"` lets the Responses API drop oldest items instead of failing. |

---

### Tier 2 — GPT-5 Specific

| Field | Type | What it controls |
|---|---|---|
| `reasoning` | `Reasoning \| None` | Controls internal chain-of-thought before answering. Imported from `openai.types.shared`. Effort values: `"none"`, `"low"`, `"medium"`, `"high"`, `"xhigh"`. SDK defaults for `gpt-5.4-mini` and `gpt-5.5`: `effort="none"`. |
| `verbosity` | `"low" \| "medium" \| "high" \| None` | Constrains the length and detail of the model's response. `"low"` is the recommended default for all agent loops. |

---

### Tier 3 — Advanced / Production

| Field | Type | What it controls |
|---|---|---|
| `store` | `bool \| None` | Store the response server-side for later retrieval. The Responses API enables this automatically when not specified. |
| `prompt_cache_retention` | `"in_memory" \| "24h" \| None` | Extend prompt cache lifetime. Set to `"24h"` to keep cached prefixes active for up to 24 hours. |
| `response_include` | `list \| None` | Request richer response payloads (e.g., usage details, token log-probs). |
| `context_management` | `list \| None` | Server-side context compaction configuration. |
| `retry` | `ModelRetrySettings \| None` | Runner-managed retry settings for model calls. |
| `extra_args` | `dict \| None` | Provider-specific fields not yet exposed at the top level. Merged (not replaced) during `resolve()`. |
| `metadata` | `dict[str, str] \| None` | Metadata included with the model response call. |

---

> **This notebook focuses on Tier 1 and Tier 2.** Tier 3 fields are production tooling — we note them here for completeness but defer them to later sections.

## Cell 5 — `temperature`: Observing the Effect

`temperature` is one of the most commonly tuned parameters. It controls the randomness of the model's token selection:

| Value | Behaviour | Best for |
|---|---|---|
| `0.0` | Near-deterministic — same input reliably produces the same output | Classification, extraction, structured tasks |
| `0.5–0.9` | Balanced — some variation, still mostly coherent | General Q&A, summaries |
| `1.0–1.4` | Creative — higher diversity, more surprising outputs | Brainstorming, copywriting, taglines |

We create two agents — one with `temperature=0.0` and one with `temperature=1.4` — and run them against the same prompt. On GPT-5 models, `temperature` works alongside `reasoning.effort`; the `effort` controls how much the model thinks before answering, while `temperature` affects the token selection once it starts generating.

> **Note on `reasoning` and `verbosity`:** We explicitly set `reasoning=Reasoning(effort="none")` and `verbosity="low"` on both agents. This matches the SDK default for `gpt-5.4-mini` and is the correct baseline for low-latency agent loops. Always set these explicitly when configuring other `ModelSettings` fields so your intent is clear.

In [7]:
agent_precise = Agent(
    name="Precise Agent",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        temperature=0.0,
    ),
)

agent_creative = Agent(
    name="Creative Agent",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        temperature=1.4,
    ),
)

prompt = "Write a one-sentence tagline for a coffee shop."

result_precise = await Runner.run(agent_precise, prompt)
result_creative = await Runner.run(agent_creative, prompt)

print("Precise:", result_precise.final_output)
print("Creative:", result_creative.final_output)

Precise: Where every cup feels like a warm welcome.
Creative: Fresh brews, warm vibes, and your perfect coffee moment.


## Cell 6 — `max_tokens`: Controlling Output Length

`max_tokens` sets a hard ceiling on how many tokens the model generates. Once the limit is reached, the model stops — it does not know it is being capped and does not produce a summary or closing statement. It simply cuts off.

This makes `max_tokens` useful in two situations:

1. **Cost control** — in agent loops where responses accumulate, short responses mean lower token bills.
2. **Enforcing conciseness** — if you want your agent to give short answers by design, `max_tokens` is a reliable backstop even when `verbosity="low"` is set.

In this cell we cap output at 20 tokens and ask for a detailed explanation of quantum computing. The response will be cut short — that is expected and intentional.

In [8]:
agent_capped = Agent(
    name="Capped Agent",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        max_tokens=20,
    ),
)

result = await Runner.run(
    agent_capped,
    "Explain quantum computing in detail.",
)

print(result.final_output)

Quantum computing is a way of processing information using **quantum mechanics**, the physics of


## Cell 7 — `frequency_penalty` and `presence_penalty` (Conceptual)

Both of these fields reduce unwanted repetition, but they do so in different ways. The distinction matters when you are tuning long-form outputs.

### `frequency_penalty`

Penalises a token in proportion to how many times it has already appeared in the output. The more often the model has used a word, the less likely it is to use it again.

- **Best for:** Long outputs where the model tends to repeat phrases or sentence structures.
- **Typical range:** `0.0–2.0`. A value of `0.5` is a mild nudge; `1.5` is aggressive.

### `presence_penalty`

Penalises any token that has appeared in the output at all, regardless of frequency. Once a word has been used, it is treated as "covered" and the model is encouraged to move on to new vocabulary.

- **Best for:** Creative tasks where you want the model to explore a wider range of ideas rather than looping back to the same concepts.
- **Typical range:** `0.0–2.0`.

### Key distinction

| Parameter | Penalises | Effect |
|---|---|---|
| `frequency_penalty` | Tokens proportional to their use count | Reduces repeated phrases and filler words in long outputs |
| `presence_penalty` | Any token that has appeared at all | Encourages the model to introduce new topics and vocabulary |

> **GPT-5 note:** On GPT-5 models, these parameters may have reduced effect compared to older GPT-4 models, because `reasoning.effort` provides a more powerful mechanism for shaping output quality. They are still useful, particularly for very long generations.

## Cell 8 — `tool_choice`: Three Values Explained (Conceptual)

`tool_choice` controls whether the model is allowed to call tools, required to call tools, or prohibited from calling tools during a given turn.

### The three string values

| Value | Meaning |
|---|---|
| `"auto"` | The model decides whether to call a tool. **This is the default behaviour** when `tool_choice` is not specified. |
| `"required"` | The model must call at least one tool. If no tool is called, the SDK raises an error. Useful when your agent must always produce a structured function call rather than a free-text response. |
| `"none"` | The model must not call any tools. The response will be a plain text generation. Useful when you want to suppress tool use on a specific turn. |

### Forcing a specific tool

You can also pass the **name of a specific tool** as a string (e.g., `tool_choice="lookup_order"`). This forces the model to call that exact tool on the next turn, rather than choosing among all available tools.

> **Note:** `tool_choice` has no practical effect right now because we have not added any tools to our agents. We are introducing it here so you understand the setting before we reach Section 3, where tools are in play. We will revisit `tool_choice` in Lecture 2.7 and demonstrate forced tool calls with real tools in Section 3.

## Cell 9 — `reasoning` and `verbosity`: GPT-5 Specifics

For GPT-5 models, `reasoning` and `verbosity` are the most important `ModelSettings` fields.

### How `reasoning.effort` works

`reasoning.effort` controls how many internal reasoning tokens the model spends before generating a response. These tokens are invisible — they happen before `final_output` and do not appear in the result. The more effort, the more internal thinking, the higher the latency and cost.

| Effort | Best for |
|---|---|
| `"none"` | Agent loops, classification, fast retrieval — latency-critical tasks where reasoning adds no value |
| `"low"` | Most tasks — a step up from `"none"` with minimal cost impact |
| `"medium"` | Balanced default for `gpt-5.5` — good starting point for general use |
| `"high"` | Complex multi-step planning, hard analysis, tasks where you have measured `"low"` underperforming |
| `"xhigh"` | Hardest async tasks, frontier evals, research-level problems |

> **Key principle:** `effort` is a cost and latency dial, not a correctness dial. On well-defined problems, `"low"` and `"high"` produce the same answer. The difference shows up in evals across many runs on genuinely hard tasks — not in a single notebook cell. Start at `"none"` or `"low"` and only move up when your evals give you a concrete reason to.

### `verbosity`

Constrains how long and detailed the model's response is:

| Value | Effect |
|---|---|
| `"low"` | Short, direct responses. **The correct default for every agent loop.** |
| `"medium"` | Balanced length. |
| `"high"` | Verbose, detailed responses. |

In an agent loop, you do not want the model padding its responses when it is one hop in a multi-agent chain. `verbosity="low"` keeps each turn tight and cost-efficient.

No code cell here — `reasoning.effort` does not produce visibly different outputs on well-defined problems. The correct mental model is the table above, not a side-by-side comparison.

## Cell 10 — `ModelSettings.resolve()`: How Settings Layer

One of the most useful — and easy to miss — behaviours in the SDK is how agent-level and `RunConfig`-level `ModelSettings` interact.

### The layering rule

When you call `Runner.run(agent, input, run_config=RunConfig(model_settings=...))`, the SDK calls `agent.model_settings.resolve(run_config.model_settings)` internally. The result is a new `ModelSettings` where:

- All **non-None** values from `RunConfig.model_settings` **override** the agent-level values.
- Fields that are `None` in `RunConfig.model_settings` keep the agent-level values.
- `extra_args` dictionaries are **merged** (not replaced).

This means you can define sensible defaults on your agent, then override individual settings per-run without reconstructing the agent.

### Resolution order

```
SDK implicit defaults
    ↓
agent.model_settings
    ↓ (non-None values from RunConfig win)
RunConfig.model_settings
```

In this cell, the agent is defined with `temperature=0.0`. We override it at run time with `temperature=1.4` via `RunConfig`. The output should be noticeably more creative than a `temperature=0.0` run.

> **RunConfig note:** `RunConfig` is covered in depth in Section 4. For now, the important thing is understanding the layering principle — knowing this saves hours of debugging in multi-agent systems.

In [12]:
base_agent = Agent(
    name="Base Agent",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
        temperature=0.0,
    ),
)

result = await Runner.run(
    base_agent,
    "Write a one-sentence tagline for a coffee shop.",
    run_config=RunConfig(
        model_settings=ModelSettings(temperature=1.4),
    ),
)

print(result.final_output)

Bold brews, cozy vibes, and your daily dose of delight.


## Cell 11 — Practical Reference: When to Use What

Use this table as a quick reference when configuring agents in your own projects.

| Scenario | Recommended setting |
|---|---|
| Deterministic output (classification, extraction) | `temperature=0.0` |
| Creative writing, brainstorming | `temperature=1.0–1.4` |
| Hard reasoning or maths | `reasoning=Reasoning(effort="high")` |
| Low-latency agent loop | `reasoning=Reasoning(effort="none"), verbosity="low"` |
| Cap cost on long tasks | `max_tokens=N` |
| Force agent to always use a tool | `tool_choice="required"` |
| Prevent context overflow | `truncation="auto"` |
| Reduce repetition in long outputs | `frequency_penalty=0.5` |

---

**Next lecture:** 2.3 — Writing effective system instructions: dos and don'ts.